In [ ]:
# CS-340 Project Two - Grazioso Salvare Dashboard
# Enhanced for CS-499 Milestone Three (Algorithms and Data Structures)

from jupyter_dash import JupyterDash
JupyterDash.infer_jupyter_proxy_config()

import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output

import base64, os, json
import pandas as pd
from functools import lru_cache

from CRUD_Python_Module import AnimalShelter

# ------------------------------------------------------------------
# Connection setup using environment variables
# ------------------------------------------------------------------

username = os.getenv("AAC_USERNAME")
password = os.getenv("AAC_PASSWORD")
auth_db = os.getenv("AAC_AUTH_DB", "admin")

if not username or not password:
    raise ValueError("Missing MongoDB credentials. Set AAC_USERNAME and AAC_PASSWORD.")

db = AnimalShelter(username, password, auth_db=auth_db)

# ------------------------------------------------------------------
# Safe MongoDB → DataFrame conversion with validation
# ------------------------------------------------------------------

def mongo_to_df(query: dict) -> pd.DataFrame:
    if not isinstance(query, dict):
        return pd.DataFrame()

    docs = db.read(query)
    if not docs:
        return pd.DataFrame()

    try:
        df = pd.DataFrame.from_records(list(docs))
    except Exception:
        return pd.DataFrame()

    if "_id" in df.columns:
        df = df.drop(columns=["_id"])

    df = df.convert_dtypes()
    return df

# ------------------------------------------------------------------
# Caching to reduce repeated MongoDB reads
# ------------------------------------------------------------------

@lru_cache(maxsize=32)
def cached_query(query_str: str):
    query = json.loads(query_str)
    return mongo_to_df(query)

# ------------------------------------------------------------------
# Filter configuration (replaces hardcoded lists)
# ------------------------------------------------------------------

FILTER_CONFIG = {
    "water": {
        "breeds": ["Labrador Retriever", "Chesapeake Bay Retriever", "Newfoundland"],
        "sex": "Intact Female",
        "min_weeks": 26,
        "max_weeks": 156
    },
    "mountain": {
        "breeds": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"],
        "sex": "Intact Male",
        "min_weeks": 26,
        "max_weeks": 156
    },
    "disaster": {
        "breeds": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"],
        "sex": "Intact Male",
        "min_weeks": 20,
        "max_weeks": 300
    }
}

def build_filter_query(filter_value: str) -> dict:
    cfg = FILTER_CONFIG.get(filter_value)
    if not cfg:
        return {}
    return {
        "breed": {"$in": cfg["breeds"]},
        "sex_upon_outcome": cfg["sex"],
        "age_upon_outcome_in_weeks": {"$gte": cfg["min_weeks"], "$lte": cfg["max_weeks"]}
    }

# ------------------------------------------------------------------
# Initial DataFrame
# ------------------------------------------------------------------

df = mongo_to_df({})

# ------------------------------------------------------------------
# Dashboard layout
# ------------------------------------------------------------------

app = JupyterDash(__name__)

logo_path = "GraziosoSalvareLogo.png"
if os.path.exists(logo_path):
    encoded = base64.b64encode(open(logo_path, "rb").read()).decode()
    logo_img = html.Img(src=f"data:image/png;base64,{encoded}", style={"height": "70px"})
else:
    logo_img = html.Div("Grazioso Salvare", style={"fontWeight": "600", "fontSize": "22px"})

header_bar = html.Div(
    [
        logo_img,
        html.Div("CS-340 Dashboard • Emily Murphy", style={"fontSize": "18px", "marginTop": "10px"})
    ],
    style={"display": "flex", "gap": "16px", "alignItems": "center"}
)

app.layout = html.Div(
    [
        header_bar,
        html.Hr(),

        html.Div(
            [
                html.Label("Rescue Type Filter", style={"fontWeight": "600"}),
                dcc.RadioItems(
                    id="filter-type",
                    options=[
                        {"label": "All", "value": "all"},
                        {"label": "Water Rescue", "value": "water"},
                        {"label": "Mountain or Wilderness", "value": "mountain"},
                        {"label": "Disaster or Individual Tracking", "value": "disaster"},
                    ],
                    value="all",
                    labelStyle={"display": "inline-block", "marginRight": "16px"}
                ),
            ],
            style={"marginBottom": "8px"}
        ),

        html.Hr(),

        dash_table.DataTable(
            id="datatable-id",
            columns=[{"name": i, "id": i} for i in df.columns],
            data=df.to_dict("records"),
            page_size=10,
            filter_action="native",
            sort_action="native",
            sort_mode="multi",
            row_selectable="single",
            selected_rows=[],
            style_table={"overflowX": "auto"},
            style_cell={"fontSize": 12, "textAlign": "left"},
        ),

        html.Br(),
        html.Hr(),

        html.Div(
            className="row",
            style={"display": "flex", "gap": "16px"},
            children=[
                html.Div(id="graph-id", style={"flex": 1}),
                html.Div(id="map-id", style={"flex": 1}),
            ],
        ),
    ],
    style={"padding": "16px"}
)

# ------------------------------------------------------------------
# Callbacks
# ------------------------------------------------------------------

@app.callback(
    Output("datatable-id", "data"),
    [Input("filter-type", "value")]
)
def update_table(filter_type):
    q = build_filter_query(filter_type if filter_type != "all" else "")
    dff = cached_query(json.dumps(q))
    return dff.to_dict("records")

@app.callback(
    Output("graph-id", "children"),
    [Input("datatable-id", "derived_virtual_data")]
)
def update_chart(view_data):
    if not view_data:
        return html.Div("No data to chart.")

    dff = pd.DataFrame(view_data)
    if dff.empty or "breed" not in dff.columns:
        return html.Div("Chart cannot be generated from empty or invalid data.")

    fig = px.pie(dff, names="breed", title="Breed Distribution")
    return dcc.Graph(figure=fig)

@app.callback(
    Output("datatable-id", "style_data_conditional"),
    [Input("datatable-id", "selected_columns")]
)
def highlight_cols(selected_columns):
    return [{"if": {"column_id": i}, "background_color": "#D2F3FF"} for i in (selected_columns or [])]

@app.callback(
    Output("map-id", "children"),
    [Input("datatable-id", "derived_virtual_data"),
     Input("datatable-id", "derived_virtual_selected_rows")]
)
def update_map(view_data, selected_rows):
    if not view_data:
        return html.Div("No rows to map.")

    dff = pd.DataFrame(view_data)
    if dff.empty:
        return html.Div("Map cannot be generated from empty data.")

    row = 0 if not selected_rows else selected_rows[0]

    lat_candidates = [c for c in dff.columns if "lat" in c.lower()]
    lon_candidates = [c for c in dff.columns if "lon" in c.lower() or "lng" in c.lower()]

    if not lat_candidates or not lon_candidates:
        return html.Div("Missing coordinate data for mapping.")

    lat_col = lat_candidates[0]
    lon_col = lon_candidates[0]

    try:
        lat = float(dff.iloc[row][lat_col])
        lon = float(dff.iloc[row][lon_col])
    except Exception:
        lat, lon = 30.75, -97.48

    return [
        dl.Map(
            style={"width": "100%", "height": "500px"},
            center=[lat, lon],
            zoom=10,
            children=[
                dl.TileLayer(),
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(str(dff.iloc[row].get("breed", "Unknown"))),
                        dl.Popup([html.H1("Animal Name"), html.P(str(dff.iloc[row].get("name", "Unknown")))])
                    ],
                ),
            ],
        )
    ]

# ------------------------------------------------------------------
# Run the dashboard
# ------------------------------------------------------------------

app.run_server(mode="inline", host="0.0.0.0", port=8060, debug=False)
